# Issue #234 — Source registry, canonical identity & temporal store schema

BJ-UNIFIED-DATA-SOURCE-SCHEMA-001 / v2 (EV R1 rework, candidate mode).

This notebook is a **read-only census** of every registered source database
and the referenced immutable packages (#203/#214/#229). It profiles tables,
keys, time ranges, nulls, duplicates, source clocks and raw-content hashes.
It **never reads settlement/label values** (C4 sealing): columns such as
`score`, `result_code`, `winning_sp`, `hit`, settlement tables are sealed
and physically unreadable through the census API.

Deliverables bound here (REQ-DS07):
- `tables/databases.csv`, `tables/tables.csv`, `tables/columns.csv`
- `tables/identity_checks.csv` (NULL-aware, correct primitive keys — C1)
- `tables/sealed_columns.csv` (C4 sealing evidence)
- `tables/source_registry.csv`, `tables/feasibility_matrix.csv` (C2)
- `MANIFEST.json` source hashes


In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd()
# When executed via nbclient from notebooks/research/issue-234, repo root is ../../..
if not (ROOT / "src").exists():
    ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "src"))
print("ROOT:", ROOT)

ROOT: /home/drwho1985/beijing-lot-234-r2


In [2]:
from source_schema import contract, ddl, profiler, sources, verify, builder
import pandas as pd
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
print("contract:", contract.CONTRACT_VERSION, "/", contract.CONTRACT_MODE)
print("data root:", profiler.resolve_data_root())

contract: BJ-UNIFIED-DATA-SOURCE-SCHEMA-001 / candidate
data root: /home/drwho1985/workspace/soccer/beijing-lot/data


In [3]:
# C4 sealing guard: settlement/label columns are physically unreadable
from source_schema.profiler import SealedColumnError, assert_measurable
sealed_attempts = [
    ("beidan_xlsm.sqlite", "matches", "result_label"),
    ("beidan_xlsm.sqlite", "match_features", "result_code"),
    ("beidan_live.db", "settlements", "winning_sp"),
    ("500_bjdc.sqlite", "matches", "result_code"),
]
for db, table, col in sealed_attempts:
    try:
        assert_measurable(db, table, col)
        print(f"UNSEALED?! {db}.{table}.{col}")
    except SealedColumnError:
        print(f"SEALED OK: {db}.{table}.{col}")
assert_measurable("beidan_live.db", "snapshots", "snapshot_ts")
print("allow-listed snapshot_ts readable OK")

SEALED OK: beidan_xlsm.sqlite.matches.result_label
SEALED OK: beidan_xlsm.sqlite.match_features.result_code
SEALED OK: beidan_live.db.settlements.winning_sp
SEALED OK: 500_bjdc.sqlite.matches.result_code
allow-listed snapshot_ts readable OK


In [4]:
# Read-only census of all registered databases
census = profiler.run_census(profiler.resolve_data_root())
dbs = pd.DataFrame(census.databases)
dbs[['database_key','exists','size_mb','sha256']].head(20)

,database_key,exists,size_mb,sha256
0,beidan_xlsm.sqlite,True,6872.9,7f62079b18b3c028884da92aa8393f5b198246c9d7c32c...
1,beidan_features.db,True,47.2,48d73b868235a78d73ea2587d5a2cd9094a861b605bfcd...
2,beidan_history.db,True,6.8,867e11452306c29e7b64b97af82f730ab0d96f7519d995...
3,beidan_live.db,True,263.4,080f7faca11dd672f31be9353c26f75846d703355037ea...
4,500_bjdc.sqlite,True,17188.7,8d990cb3f36a16f2db78fe6050a718235b275b5d6af802...
5,beidan_modeling_plus_2026.sqlite,True,1087.8,c35a42800a7998ede97c5105371cd78a293a151a9c004a...
6,beidan_nowscore_2026.sqlite,True,191.6,3948176d42e7799a986e406a75266fe840eadf977c21d8...


In [5]:
tables = pd.DataFrame(census.tables)
tables.head(30)

,database_key,table,type,rows,n_columns,sealed
0,beidan_xlsm.sqlite,action_candidates,table,1562152,19,True
1,beidan_xlsm.sqlite,action_definitions,table,8,4,False
2,beidan_xlsm.sqlite,exotic_odds,table,7615491,4,False
3,beidan_xlsm.sqlite,issue_list,table,27,1,False
4,beidan_xlsm.sqlite,line_lookup,table,153,9,False
5,beidan_xlsm.sqlite,market_consensus,table,3124304,20,False
6,beidan_xlsm.sqlite,market_snapshots,table,51551016,7,False
7,beidan_xlsm.sqlite,match_features,table,195269,41,False
8,beidan_xlsm.sqlite,matches,table,195269,25,False
9,beidan_xlsm.sqlite,official_500_settlements,table,74295,26,True


In [6]:
# C1: identity metrics — NULL-aware, correct primitive keys, real duplicate excess
identity = pd.DataFrame(census.identity)
identity[['database_key','table','primitive_key','total_rows','null_key_rows',
         'non_null_rows','distinct_non_null_keys','duplicate_excess','uniqueness_verdict']]

,database_key,table,primitive_key,total_rows,null_key_rows,non_null_rows,distinct_non_null_keys,duplicate_excess,uniqueness_verdict
0,beidan_xlsm.sqlite,match_features,issue|match_idx,195269,2560,192709,192709,0,NULL_KEY_DEFECT
1,beidan_xlsm.sqlite,matches,issue|match_idx,195269,2560,192709,192709,0,NULL_KEY_DEFECT
2,beidan_xlsm.sqlite,outcome_features,issue|match_idx|outcome_code,585807,7680,578127,578127,0,NULL_KEY_DEFECT
3,beidan_features.db,matches,match_id,190746,0,190746,190743,3,DUPLICATE_DEFECT
4,beidan_history.db,historical_matches,issue|match_idx,38463,0,38463,38463,0,UNIQUE
5,beidan_live.db,matches,issue|match_idx,5155,0,5155,5155,0,UNIQUE
6,500_bjdc.sqlite,matches,issue|match_idx,76043,0,76043,76043,0,UNIQUE
7,beidan_modeling_plus_2026.sqlite,match_features,issue|match_idx,201270,2560,198710,198710,0,NULL_KEY_DEFECT
8,beidan_modeling_plus_2026.sqlite,matches,issue|match_idx,201270,2560,198710,198710,0,NULL_KEY_DEFECT
9,beidan_modeling_plus_2026.sqlite,outcome_features,issue|match_idx|outcome_code,603810,7680,596130,596130,0,NULL_KEY_DEFECT


In [7]:
# C4: sealed columns — settlement/label values never measured
sealed = pd.DataFrame(census.sealed)
print("sealed entries:", len(sealed))
sealed.head(20)

sealed entries: 488


,database_key,table,columns,reason,column
0,beidan_xlsm.sqlite,action_candidates,"match_id,issue,match_idx,kickoff_ts,season_yea...",settlement/label truth,NaN
1,beidan_xlsm.sqlite,action_definitions,NaN,settlement/label column; not allow-listed,action_code
2,beidan_xlsm.sqlite,action_definitions,NaN,settlement/label column; not allow-listed,action_label
3,beidan_xlsm.sqlite,action_definitions,NaN,settlement/label column; not allow-listed,selection_set
4,beidan_xlsm.sqlite,action_definitions,NaN,settlement/label column; not allow-listed,cover_count
5,beidan_xlsm.sqlite,line_lookup,NaN,settlement/label column; not allow-listed,row_no
6,beidan_xlsm.sqlite,line_lookup,NaN,settlement/label column; not allow-listed,c1
7,beidan_xlsm.sqlite,line_lookup,NaN,settlement/label column; not allow-listed,c2
8,beidan_xlsm.sqlite,line_lookup,NaN,settlement/label column; not allow-listed,c3
9,beidan_xlsm.sqlite,line_lookup,NaN,settlement/label column; not allow-listed,c4


In [8]:
# Clock evidence (REQ-DS03): timestamp columns measured, semantics assigned
ts = pd.DataFrame(census.timestamps)
ts[['database_key','table','column','clock_semantics','min','max']].head(40)

,database_key,table,column,clock_semantics,min,max
0,beidan_xlsm.sqlite,match_features,kickoff_ts,SCHEDULE_ONLY,1899-12-30 08:00:00,2025-07-13 03:00:00
1,beidan_xlsm.sqlite,matches,kickoff_ts,SCHEDULE_ONLY,1899-12-30 08:00:00,2025-07-13 03:00:00
2,beidan_xlsm.sqlite,matches,ts_10h,LABEL_RELATIVE,2013-12-28 06:49:59,2025-07-12 17:00:00
3,beidan_xlsm.sqlite,matches,kickoff_serial,NO_CLOCK,0.333333333333333,45851.125
4,beidan_xlsm.sqlite,matches,ts_10h_serial,NO_CLOCK,41636.2847222222,45850.708333333336
5,beidan_xlsm.sqlite,outcome_features,kickoff_ts,SCHEDULE_ONLY,1899-12-30 08:00:00,2025-07-13 03:00:00
6,beidan_features.db,matches,kickoff,SCHEDULE_ONLY,08:00:00,2025-04-15T08:30:00
7,beidan_history.db,historical_matches,kickoff_ts,SCHEDULE_ONLY,2024-01-01 14:00:00,2026-12-31 16:00:00
8,beidan_history.db,scrape_log,fetched_at,TIMESTAMPED_LOCAL_NO_TZ,2026-05-02 00:54:48,2026-05-02 00:58:42
9,beidan_live.db,matches,kickoff_ts,SCHEDULE_ONLY,2026-04-30T19:30:00,2026-08-18T11:06:00


In [9]:
# Source registry + feasibility matrix (C2: evidence, not reputation)
census_hashes = {d['database_key']: d['sha256'] for d in census.databases if d.get('sha256')}
registry = sources.registry_frame(census_hashes)
registry[['source_id','provider','endpoint','clock_semantics','admissible_use',
         'coverage_grade','raw_sha256','selected_for_branch']]

,source_id,provider,endpoint,clock_semantics,admissible_use,coverage_grade,raw_sha256,selected_for_branch
0,BJ_FIRST_PARTY_OFFICIAL,Beijing Sports Lottery official interface (not...,official interface (未接入); no repo collector ex...,NO_CLOCK,UNUSABLE,F,NO_RECEIPT,
1,BEIDAN_LIVE_DB,internal live collector over 500.com data plane,trade.500.com/bjdc/?expect={issue} + www.500.c...,TIMESTAMPED_UTC,STRICT_CAUSAL,A,080f7faca11dd672f31be9353c26f75846d703355037ea...,BJ_SP_TIMESERIES #235
2,BEIDAN_XLSM_WORKBOOK,internal workbook conversion (beidan.xlsm),data/derived/beidan_xlsm/beidan_xlsm.sqlite,LABEL_RELATIVE,RETROSPECTIVE_PROXY,C,7f62079b18b3c028884da92aa8393f5b198246c9d7c32c...,
3,BEIDAN_XLSM_SETTLEMENTS,internal workbook conversion (official_500_set...,data/derived/beidan_xlsm/beidan_xlsm.sqlite:of...,SETTLEMENT_AFTER_EVENT,SETTLEMENT_ONLY,D,7f62079b18b3c028884da92aa8393f5b198246c9d7c32c...,
4,BEIDAN_FEATURES_DB,internal features archive,data/beidan_features.db,LABEL_RELATIVE,RETROSPECTIVE_PROXY,C,48d73b868235a78d73ea2587d5a2cd9094a861b605bfcd...,
5,BEIDAN_HISTORY_DB,internal settlement/history archive,data/beidan_history.db,SETTLEMENT_AFTER_EVENT,SETTLEMENT_ONLY,D,867e11452306c29e7b64b97af82f730ab0d96f7519d995...,
6,BEIDAN_500BJDC,internal 500.com collector (raw receipts + wor...,trade.500.com / live.500.com / zx.500.com,TIMESTAMPED_UTC,STRICT_CAUSAL,A,8d990cb3f36a16f2db78fe6050a718235b275b5d6af802...,INTEGRATION #238
7,BEIDAN_MODELING_PLUS_2026,internal modeling archive (2026),data/derived/beidan_xlsm/beidan_modeling_plus_...,LABEL_RELATIVE,RETROSPECTIVE_PROXY,C,c35a42800a7998ede97c5105371cd78a293a151a9c004a...,
8,BEIDAN_NOWSCORE_2026,internal nowscore archive (2026),data/derived/beidan_xlsm/beidan_nowscore_2026....,LABEL_RELATIVE,RETROSPECTIVE_PROXY,C,3948176d42e7799a986e406a75266fe840eadf977c21d8...,
9,ODDS_API_HISTORICAL,The Odds API,api.the-odds-api.com (historical snapshots fro...,TIMESTAMPED_UTC,RETROSPECTIVE_PROXY,B,NO_RECEIPT,EU_ODDS #236


In [10]:
# Build the immutable package and run the semantic verifier (E2)
package_dir = ROOT / "results" / contract.PACKAGE_DIR_NAME
builder.build_package(ROOT, package_dir, force=True)
verdict = verify.verify_package(package_dir, ROOT)
print("VERIFIED" if verdict["verified"] else "VERIFY FAILED")
for problem in verdict["problems"]:
    print("  -", problem)

VERIFIED


In [11]:
# REQ-DS evidence table
pd.read_csv(package_dir / "evidence" / "req_ds_evidence.csv")

,requirement,result,evidence
0,REQ-DS01,PASS,feasibility_matrix.csv: admissible_use + endpo...
1,REQ-DS02,PASS,"DDL UNIQUE(issue_code,match_idx) + identity_ch..."
2,REQ-DS03,PASS,domain_dictionary.md + DDL canonical-UTC CHECK...
3,REQ-DS04,PASS,"DDL quote PK(observation,bookmaker,outcome) + ..."
4,REQ-DS05,FAIL,parquet_schema.json disjoint inference/proxy/s...
5,REQ-DS06,PASS,tables.csv measured rows=96109984; design_note...
6,REQ-DS07,PASS,MANIFEST.json source_hashes + columns.csv + no...
7,REQ-DS08,PASS,"downstream_handoff.md selected sources: ['', '..."


## Conclusion

- Sources are classified by measured evidence (REQ-DS01); the 500.com
  third-party collector plane is named honestly (C2), and the first-party
  Beijing interface is registered UNUSABLE until acquired.
- Legacy identity conclusions are NULL-aware: 2,560 NULL-key rows in
  `beidan_xlsm.matches` are UNIDENTIFIED, not duplicates; real
  duplicate excess is 0 (C1).
- The frozen DDL rejects orphan quotes (FK ON), duplicate primitive
  observations, duplicate euro cells (COALESCE unique), overlapping alias
  windows, non-canonical/reversed timestamps, and observation-after-kickoff
  (C3). Multi-bookmaker quotes are legal.  Prospective live captures are
  expressible: `snapshot_kind='live'` is accepted for the timestamped
  collector plane (C3-5).
- 2024+ outcome/settlement values are sealed: the census never reads them,
  and the notebook proves the guard rejects sealed reads (C4).
- The semantic verifier rebuilds the schema and registry from committed
  payloads; every remanifest attack in the test suite fails closed (E2).
